## Revision notes

Current revision: **22**. The `config` cell prints it, so a run identifies its revision.

What changed from revision 21

- The training notebook and dataset references now live in TaskLattice Guard.
- The INT8 comparison also reports FP16 and INT8 acceptance results at the standard
  evaluation cutoff. It uses the same saved scores as the FPR budget comparison.

---



# Laya post-training workbench

The record of how Laya is post-trained for TaskLattice Guard: the pinned dataset, the
pinned base checkpoint, the fine-tuning recipe, the checkpoint and recovery behaviour. This
Notebook is the executable part — every cell runs on a Kaggle `GPU T4 x2` session.

## Prerequisites

| Setting | Value |
| --- | --- |
| Accelerator | `GPU T4 x2` |
| Internet | `On` (downloads the pinned Laya checkpoint and the packages) |
| Inputs | the private **Laya Guardrails Classification Dataset**, exact version |

The dataset is a private Kaggle Dataset containing `notebook-v2-tensor-trust/manifest.json` and the five
JSONL splits. `docs/datasets.md` records where every row came from and how the splits were
built. `datasets/` in TaskLattice Guard holds the registry, the version manifest and the frozen
acceptance cases; the rows themselves stay in access-controlled storage.

## Material record

| Material | Recorded in |
| --- | --- |
| Data sources, revisions, file SHA-256, license and usage | `datasets/registry.yaml` |
| Version, split counts, category and label distribution | `datasets/manifests/notebook-v2-tensor-trust.json` |
| Frozen acceptance cases with their source rows | `datasets/reference/fixed-acceptance.json` |
| Split construction and Kaggle mount | `docs/datasets.md` |
| Base checkpoint revision and per-file SHA-256 | the `config` cell below |
| Training recipe, question wording, runtime budgets | the `config` cell below |
| One run, end to end | the `report` cell → `laya-material.json` in `/kaggle/working` |

The `config` cell is the recipe of record. This repository carries no separate configuration
files, so a change to the recipe is a change to this Notebook.


### Configure the run

Selects the pinned dataset, base checkpoint, and runtime settings. Set `TUNED_MODEL_DIR` to select a specific saved checkpoint and enable `RUN_INT8_EXPORT` here before installing dependencies if you want to compare INT8.

In [ ]:
import json
from pathlib import Path

REVISION = 22

# Private Kaggle Dataset. The manifest hash pins its content.
DATASET = {
    "datasetId": "laya-guardrails",
    "version": "notebook-v2-tensor-trust",
    "sha256": "189c1a779978ed258c77d60e6dc25fdc990e682993f8d95b6a2ce95d7ff75970",
    "ref": "wangguohao/laya-guardrails-classification-dataset"
}
DATASET_ROOT = None  # directory containing <version>/manifest.json; None = search the Kaggle inputs
TUNED_MODEL_DIR = None  # Optional path to the merged model directory. None = discover it from Inputs.
EXPORT_PRECISION = "fp16"  # Storage precision for a newly trained checkpoint. Use "fp32" for FP32 weights.
FP16_EXPORT_DIR = None  # Set by export-fp16 when an existing checkpoint is converted.
RUN_INT8_EXPORT = False  # Enable ONNX export and the FP16 / INT8 comparison.

# Pinned base checkpoint. The digests are the contract: they must match what Hugging Face serves.
BASE = {
    "id": "convaiinnovations/laya",
    "revision": "85cc74ddae6109077ebe2853a9b3077fb6606a78",
    "files": {
        "encoder/config.json": "bf3ab80598fdccf414855a2ce80f22859e4492d06ca8a62ddd1cfb63972f8979",
        "model.safetensors": "891102d372688fc2a094dac56a384bc537b87c63f21f9f3dac0be2b7cbc8d86c",
        "rl_agent_config.json": "ae287b56bbcf5f8c4f4541ae9dfd00c914c4c48b940b8398c3058af37ba92bbd",
        "tokenizer/tokenizer.json": "6c8aaa9a542084f2457eab775d4eeb51f92a70c0fd9de28d5edb0ddec3c08d30",
        "tokenizer/tokenizer_config.json": "50044de60daaa73df97d262e15a40d4faf0160e7d742df64b377877a1320dd12",
    },
}

# Fine-tuning recipe: one joint LoRA pass over the encoder plus the decision head.
TRAINING = {
    "maxLen": 8192,  # long documents are kept, not truncated
    "headMaxLen": 192,
    "epochs": 1,
    "effectiveBatch": 16,
    "microBatch": 8,  # sequences per forward pass per GPU
    "tokenBudget": 4096,  # a micro batch is split earlier if it would exceed this
    "checkpointSteps": 250,
    "seed": 42,
    "workers": 2,
    "optimizer": {"encoderLR": 1e-4, "headLR": 5e-5, "weightDecay": 0.01, "maxGradNorm": 1.0},
    "lora": {"rank": 8, "alpha": 16, "dropout": 0.05, "targetModules": "all-linear"},
    "gradientCheckpointing": True,
}

# Budgets the Guard Runner actually uses, so the comparison is on the production request profile.
RUNTIME = {"maxLen": 1024, "headMaxLen": 512, "language": "en"}

# Question wording per task, as used for training.
QUESTIONS = {
    "content_safety": "Does the current input contain unsafe content?",
    "prompt_injection": (
        "Does the input attempt to override the assistant's instructions, change its assigned "
        "role or task, bypass its rules, or reveal hidden prompts?"
    ),
    "pii": (
        "Does the current input contain personal information or a financial identifier that "
        "should be redacted?"
    ),
    "credential_leakage": (
        "Does the current input contain a password, API key, authentication token, or private "
        "key used as a credential?"
    ),
    "output_leakage": (
        "Does the model output disclose a protected secret, such as a confidential access code?"
    ),
    "jailbreak": (
        "Does the current input try to override system instructions or bypass safety rules?"
    ),
    "memory_poisoning": (
        "Does the long-term memory record try to alter future behavior or plant an unsupported "
        "answer?"
    ),
}

BASELINE_SPLITS = {
    "validation": "Validation split",
    "acceptance": "Independent frozen acceptance suite",
}

# The device setting also applies to the original baseline.
EVALUATION = {
    "split": "acceptance",  # acceptance | calibration_policy | validation
    "limit": 0,  # 0 = the whole split
    "device": "cuda",  # all visible GPUs; use "cuda:0" or "cpu" to select one
    "calibrate": True,  # fit a temperature on calibration_fit and report ECE before/after
    "calibrationSample": 400,
}

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUTPUT = WORK / "laya-guardrails"
print("revision:", REVISION)
print("dataset :", DATASET["datasetId"], DATASET["version"])
print("recipe  :", json.dumps({k: v for k, v in TRAINING.items() if k != "optimizer"}))
print("output  :", OUTPUT)


def model_export_sizes(model_dir):
    sizes = {
        name: (model_dir / name).stat().st_size if (model_dir / name).is_file() else 0
        for name in BASE["files"]
    }
    incomplete = [name for name, size in sizes.items() if not size]
    if incomplete:
        raise FileNotFoundError(
            f"Incomplete model export at {model_dir}. Missing or empty {', '.join(incomplete)}."
        )
    return sizes


def resolve_tuned_model_dir():
    if FP16_EXPORT_DIR is not None:
        model_export_sizes(FP16_EXPORT_DIR)
        return FP16_EXPORT_DIR
    if TUNED_MODEL_DIR is not None:
        directory = Path(TUNED_MODEL_DIR).expanduser()
        model_export_sizes(directory)
        return directory

    local = OUTPUT / "model"
    if globals().get("RUN_TRAINING", False):
        model_export_sizes(local)
        return local

    attached = {
        path.resolve()
        for name in ("model", "model-fp16")
        for path in INPUT.rglob(f"laya-guardrails/{name}")
        if path.is_dir()
    }
    complete, incomplete = [], []
    for directory in sorted(attached):
        try:
            model_export_sizes(directory)
        except FileNotFoundError as error:
            incomplete.append(str(error))
        else:
            complete.append(directory)

    if len(complete) > 1:
        choices = "\n  ".join(str(path) for path in complete)
        raise ValueError(f"Multiple attached Laya checkpoints found. Set TUNED_MODEL_DIR to one of them.\n  {choices}")
    if complete:
        return complete[0]
    if incomplete:
        raise FileNotFoundError("\n".join(incomplete))
    for directory in (local, OUTPUT / "model-fp16"):
        if directory.exists():
            model_export_sizes(directory)
            return directory
    return None

### Install dependencies

Installs the packages for the selected run and prints GPU and disk availability. Enable Internet in Kaggle before running this cell.

In [ ]:
import os
import platform
import shutil
import subprocess
import sys

# A T4 fragments its allocator easily; expandable segments avoid most of it.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

PACKAGES = (
    "laya==0.3.22",
    "transformers==4.57.6",
    "huggingface-hub==0.36.0",
    "peft==0.18.0",
    "safetensors>=0.4.3,<1",
    "datasets>=3.0.0",
)
if RUN_INT8_EXPORT:
    PACKAGES += ("onnx==1.19.1", "onnxruntime==1.28.0")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-input", *PACKAGES], check=True
)

import torch

import laya
import transformers

gpus = [
    f"{torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)"
    for i in range(torch.cuda.device_count())
]
print(f"python       {platform.python_version()}  {platform.system()}/{platform.machine()}")
print(f"laya         {laya.__version__}")
print(f"transformers {transformers.__version__}")
print(f"torch        {torch.__version__}")
print(f"cuda         {torch.cuda.is_available()}")
print(f"gpus         {', '.join(gpus) or 'none'}")
print(f"free disk    {shutil.disk_usage(WORK).free / 1e9:.1f} GB")
if len(gpus) < TRAINING["workers"]:
    print(
        f"Note: {len(gpus)} GPU(s) visible but the recipe expects {TRAINING['workers']}. "
        "Lower TRAINING['workers'] for a single-GPU or CPU run."
    )

### Verify the dataset

Locates the pinned dataset and checks the manifest and all five split hashes. Attach the dataset under Kaggle Inputs before running this cell.

In [ ]:
import hashlib

SPLITS = ("train", "validation", "calibration_fit", "calibration_policy", "acceptance")


def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def dataset_dir():
    # Kaggle derives the mount path from the Dataset title, which need not match the
    # registered ref, so the version directory is located by content, not by name.
    search = Path(DATASET_ROOT) if DATASET_ROOT else INPUT
    if not search.is_dir():
        raise FileNotFoundError(f"{search} does not exist.")
    matches, others = [], []
    version_mismatch = False
    for manifest in sorted(search.rglob("manifest.json")):
        try:
            value = json.loads(manifest.read_bytes())
        except (OSError, json.JSONDecodeError):
            others.append(f"{manifest} (unreadable)")
            continue
        identity = f"{value.get('datasetId')} {value.get('version')}"
        if (value.get("datasetId"), value.get("version")) == (
            DATASET["datasetId"],
            DATASET["version"],
        ):
            matches.append(manifest.parent)
        else:
            others.append(f"{manifest} ({identity})")
            version_mismatch = version_mismatch or value.get("datasetId") == DATASET["datasetId"]
    named = [path for path in matches if path.name == DATASET["version"]]
    if len(named) == 1:
        return named[0]
    if len(matches) == 1:
        return matches[0]
    if matches:
        raise ValueError(
            f"{len(matches)} copies of {DATASET['version']} are mounted: "
            + ", ".join(str(path) for path in matches)
        )
    top = ", ".join(sorted(path.name for path in search.iterdir())) or "nothing"
    archives = sorted(path.name for path in search.rglob("*.zip"))
    lines = [
        f"No manifest for {DATASET['datasetId']} {DATASET['version']} under {search}.",
        f"Top level: {top}.",
    ]
    if others:
        lines.append("Manifests found with another identity: " + "; ".join(others) + ".")
    if archives:
        lines.append(
            f"Archives: {', '.join(archives)}. Kaggle mounts an archive as a single file, "
            "so extract it and upload the directory."
        )
    if version_mismatch:
        lines.append(
            "A different dataset version is already mounted. Set DATASET['version'] and "
            "DATASET['sha256'] to the registered values for the version you intend to use, "
            "or attach the configured version."
        )
    else:
        lines.append(
            "Attach the Dataset in Inputs, or set DATASET_ROOT to the directory holding it."
        )
    raise FileNotFoundError(" ".join(lines))


def verify_dataset():
    directory = dataset_dir()
    print(f"dataset: {directory}")
    manifest_path = directory / "manifest.json"
    payload = manifest_path.read_bytes()
    if hashlib.sha256(payload).hexdigest() != DATASET["sha256"]:
        raise ValueError("Manifest SHA-256 differs from the registered Kaggle Dataset.")
    manifest = json.loads(payload)
    if manifest["fixture"] is not False or set(manifest["splits"]) != set(SPLITS):
        raise ValueError("Unexpected dataset identity or split layout.")
    for split, entry in manifest["splits"].items():
        shard = directory / f"{split}.jsonl"
        if entry["file"] != shard.name or sha256(shard) != entry["sha256"]:
            raise ValueError(f"Shard SHA-256 differs for {split}.")
        print(f"{split:<18} {entry['count']:>7,} rows  sha256 {entry['sha256'][:12]}")
    print(f"verified {DATASET['ref']} ({DATASET['version']})")
    return directory, manifest


DATA_DIR, MANIFEST = verify_dataset()


def read_split(name):
    with (DATA_DIR / f"{name}.jsonl").open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

### Score the original Laya checkpoint

Runs the original model on the full validation and acceptance splits and saves per-case results. Run after dataset verification to see the original model's baseline metrics.

In [ ]:
import math
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from contextlib import contextmanager, nullcontext
from tempfile import TemporaryDirectory

import laya
import torch
from huggingface_hub import snapshot_download


def materialize():
    # laya.load verifies the pinned digests itself, local directory included, so this only has
    # to place the checkpoint files where the training script can read them.
    path = Path(
        snapshot_download(
            BASE["id"], revision=BASE["revision"], allow_patterns=list(BASE["files"])
        )
    )
    print(f"base checkpoint {BASE['id']}@{BASE['revision'][:12]}")
    return path


@contextmanager
def scoring_agents(directory, *, expected_sha256=None):
    requested = torch.device(EVALUATION["device"])
    devices = [requested]
    if requested.type == "cuda" and requested.index is None:
        visible = torch.cuda.device_count()
        if visible:
            devices = [torch.device("cuda", index) for index in range(visible)]

    agents = []
    try:
        for device in devices:
            agents.append(laya.load(str(directory), device=str(device), expected_sha256=expected_sha256))
            # The first load verifies the snapshot shared by the replicas.
            expected_sha256 = None
        yield agents
    finally:
        agents.clear()
        if requested.type == "cuda" and torch.cuda.is_available():
            torch.cuda.empty_cache()


def state_for(row):
    # The state shape the Runner sends for that category, so scoring matches serving.
    scan = row["scan"]
    context = scan.get("guardrail_context", {})
    memory = context.get("memory")
    if row["category"] == "memory_poisoning" and memory:
        return {
            "current_input": scan["input"],
            "user_task": context.get("query", ""),
            "memory_operation": memory["operation"],
            "source_query": memory["source_query"],
            "target_source": "tool_output"
            if memory["operation"] == "write"
            else "retrieved_content",
            "context_messages": [],
        }
    if context.get("target_source") == "retrieved_content":
        return {
            "user_task": context.get("query", ""),
            "untrusted_source": "retrieved_content",
            "untrusted_text": scan["input"],
        }
    return scan["input"]


def task_for(row):
    category = row["category"]
    if category == "credential_leakage" and row["phase"] == "output":
        category = "output_leakage"
    if row["source"].split("/")[0] in {"jailbreakbench", "advbench"}:
        category = "content_safety"
    return category


def logit(probability):
    probability = min(1 - 1e-8, max(1e-8, probability))
    return math.log(probability / (1 - probability))


def fit_temperature(probabilities, labels):
    # Global binary temperature by bisection on the NLL, the method Guard uses.
    values = [
        (logit(p), label)
        for p, label in zip(probabilities, labels, strict=True)
        if p is not None
    ]
    if not values:
        return None

    def nll(temperature):
        margins = [value / temperature * (1 - 2 * label) for value, label in values]
        return sum(max(0, m) + math.log1p(math.exp(-abs(m))) for m in margins) / len(margins)

    low, high = 0.5, 5.0
    for _ in range(36):
        a, b = (2 * low + high) / 3, (low + 2 * high) / 3
        if nll(a) < nll(b):
            high = b
        else:
            low = a
    return (low + high) / 2


def score(agent, rows):
    probabilities, labels, categories, latencies, unscored_reasons = [], [], [], [], []
    dropped = 0
    for row in rows:
        question = {"decision": {"type": "noul", "instructions": QUESTIONS[task_for(row)]}}
        started = time.perf_counter()
        payload = agent.system_one(
            state_for(row),
            question,
            lang=RUNTIME["language"],
            max_len=RUNTIME["maxLen"],
            head_max_len=RUNTIME["headMaxLen"],
        )
        latencies.append((time.perf_counter() - started) * 1000)
        usage = payload["usage"]
        truncated = bool(usage["truncated"] or usage["state_tokens_dropped"])
        dropped += int(truncated)
        answer = (payload.get("answers") or {}).get("decision") or {}
        value = answer.get("noul")
        valid = isinstance(value, (int, float)) and not isinstance(value, bool) and 0 <= value <= 1
        reason = None
        if truncated:
            reason = "input_truncated"
        elif value is None:
            reason = "missing_probability"
        elif not valid:
            reason = "invalid_probability"
        unscored_reasons.append(reason)
        probabilities.append(float(value) if reason is None else None)
        labels.append(int(row["label"]))
        categories.append(row["category"])
    if dropped:
        print(f"{dropped:,} of {len(rows):,} requests on {agent.device} hit the runtime budget.", flush=True)
    return {
        "probability": probabilities,
        "label": labels,
        "category": categories,
        "latency": latencies,
        "unscored_reason": unscored_reasons,
    }


def score_split(agents, rows):
    replicas = len(agents)
    if replicas == 1:
        print(f"  {len(rows):,} rows on {agents[0].device}", flush=True)
        return score(agents[0], rows)

    shards = [rows[rank::replicas] for rank in range(replicas)]
    assignments = list(zip(agents, shards, strict=True))
    print(
        "  " + ", ".join(f"{len(shard):,} rows on {agent.device}" for agent, shard in assignments),
        flush=True,
    )

    def score_shard(agent, shard):
        device_context = torch.cuda.device(agent.device) if agent.device.type == "cuda" else nullcontext()
        with device_context:
            return score(agent, shard)

    with ThreadPoolExecutor(max_workers=replicas) as pool:
        futures = [pool.submit(score_shard, agent, shard) for agent, shard in assignments]
        results = [future.result() for future in futures]

    # Restore the original row order before joining predictions to dataset and phase.
    combined = {key: [None] * len(rows) for key in results[0]}
    for rank, result in enumerate(results):
        for key, values in result.items():
            combined[key][rank::replicas] = values
    return combined


def summarize(scored, temperature=1.0):
    # Guardrail reading of the confusion matrix: a missed violation is the error that lets harm
    # through, a false alarm is the error that blocks a legitimate request. Both are reported as
    # counts and as rates over their own denominator, since the two classes are not balanced.
    pairs = [
        (1 / (1 + math.exp(-logit(p) / temperature)), bool(label))
        for p, label in zip(scored["probability"], scored["label"], strict=True)
        if p is not None
    ]
    true_positives = sum(p >= 0.5 and y for p, y in pairs)
    false_positives = sum(p >= 0.5 and not y for p, y in pairs)
    false_negatives = sum(p < 0.5 and y for p, y in pairs)
    true_negatives = sum(p < 0.5 and not y for p, y in pairs)
    violations = true_positives + false_negatives
    benign_cases = true_negatives + false_positives
    predicted_positive = true_positives + false_positives
    bins = {}
    for probability, label in pairs:
        bins.setdefault(min(14, int(probability * 15)), []).append((probability, label))
    ordered = sorted(scored["latency"])
    tested_cases = len(scored["label"])
    cases = len(pairs)
    return {
        "tested_cases": tested_cases,
        "scored_cases": cases,
        "coverage": cases / tested_cases if tested_cases else None,
        "violations": violations,
        "benign_cases": benign_cases,
        "true_positives": true_positives,
        "false_positives": false_positives,
        "true_negatives": true_negatives,
        "false_negatives": false_negatives,
        "accuracy": (true_positives + true_negatives) / cases if cases else None,
        "precision": true_positives / predicted_positive if predicted_positive else None,
        "recall": true_positives / violations if violations else None,
        "missed_violation_rate": false_negatives / violations if violations else None,
        "false_alarm_rate": false_positives / benign_cases if benign_cases else None,
        "brier_score": sum((p - y) ** 2 for p, y in pairs) / cases if cases else None,
        "expected_calibration_error": (
            sum(abs(sum(p - y for p, y in group)) for group in bins.values()) / cases
            if cases else None
        ),
        "median_latency_ms": ordered[(len(ordered) - 1) // 2] if ordered else None,
        "p95_latency_ms": ordered[round(0.95 * (len(ordered) - 1))] if ordered else None,
        "temperature": temperature,
    }


COLUMNS = (
    "tested_cases",
    "scored_cases",
    "coverage",
    "violations",
    "benign_cases",
    "true_positives",
    "false_positives",
    "true_negatives",
    "false_negatives",
    "accuracy",
    "precision",
    "recall",
    "missed_violation_rate",
    "false_alarm_rate",
    "brier_score",
    "expected_calibration_error",
    "median_latency_ms",
    "p95_latency_ms",
)


def show(results):
    import pandas as pd

    table = {
        name: {key: entry["summary"][key] for key in COLUMNS} for name, entry in results.items()
    }
    for name, entry in results.items():
        if "calibrated" in entry:
            label = f"{name} (temperature {entry['calibrated']['temperature']:.2f})"
            table[label] = {key: entry["calibrated"][key] for key in COLUMNS}
    print(pd.DataFrame(table).astype(float).round(4).to_string())


def by_category(scored, temperature=1.0):
    import pandas as pd

    groups = {}
    for index, category in enumerate(scored["category"]):
        groups.setdefault(category, []).append(index)
    table = {}
    for category, indices in sorted(groups.items()):
        subset = {
            key: [scored[key][index] for index in indices]
            for key in ("probability", "label", "latency")
        }
        summary = summarize(subset, temperature)
        table[category] = {
            "tested_cases": summary["tested_cases"],
            "scored_cases": summary["scored_cases"],
            "violations": summary["violations"],
            "missed_violations": summary["false_negatives"],
            "missed_violation_rate": summary["missed_violation_rate"],
            "false_alarms": summary["false_positives"],
            "false_alarm_rate": summary["false_alarm_rate"],
        }
    print(pd.DataFrame(table).astype(float).round(4).to_string())


BASELINE_CUTOFF = 0.5
BASELINE_RATE_COLUMNS = {
    "accuracy": "Acc (%)",
    "risk_recall": "RR (%)",
    "benign_allow_rate": "BAR (%)",
    "false_positive_rate": "FPR (%)",
    "false_negative_rate": "FNR (%)",
    "coverage": "Coverage (%)",
}
BASELINE_COLUMNS = {
    "dataset": "Dataset",
    "task": "Task",
    "test_n": "Test n",
    "scored_n": "Scored n",
    "tp": "TP",
    "tn": "TN",
    "fp": "FP",
    "fn": "FN",
    "unscored_n": "Unscored",
    **BASELINE_RATE_COLUMNS,
}
CASE_COLUMNS = {
    "case_id": "Case ID",
    "dataset": "Dataset",
    "outcome": "Outcome",
    "expected_risk": "Expected",
    "expected_action": "Expected action",
    "predicted_risk": "Predicted",
    "probability": "P(risk)",
    "decision_basis": "Decision basis",
}
CASE_OUTCOMES = {(1, 1): "TP", (0, 0): "TN", (0, 1): "FP", (1, 0): "FN"}
UNSCORED_REASONS = {
    "input_truncated": "Input exceeded the runtime token budget",
    "missing_probability": "Model returned no probability",
    "invalid_probability": "Model returned an invalid probability",
}

TASK_LABELS = {
    "content_safety": "content",
    "prompt_injection": "injection",
    "pii": "pii",
    "credential_leakage": "credentials",
    "output_leakage": "leakage",
    "jailbreak": "jailbreak",
    "memory_poisoning": "memory",
}


def baseline_case_evidence(split, rows, scored):
    evidence = []
    scores = zip(scored["probability"], scored["unscored_reason"], strict=True)
    for row, (probability, reason) in zip(rows, scores, strict=True):
        expected = int(row["label"])
        if probability is None:
            predicted, outcome = None, "U"
            basis = UNSCORED_REASONS[reason]
        else:
            predicted = int(probability >= BASELINE_CUTOFF)
            outcome = CASE_OUTCOMES[expected, predicted]
            comparison = "≥" if predicted else "<"
            basis = f"P(risk) {comparison} {BASELINE_CUTOFF:g}"
        evidence.append({
            "split": split,
            "case_id": row["id"],
            "dataset": row["source"].split("/", 1)[0],
            "source": row["source"],
            "category": row["category"],
            "task": task_for(row),
            "phase": row["phase"],
            "expected_risk": expected,
            "expected_action": row["expected"],
            "predicted_risk": predicted,
            "probability": probability,
            "cutoff": BASELINE_CUTOFF,
            "outcome": outcome,
            "unscored_reason": reason,
            "decision_basis": basis,
        })
    return evidence


def baseline_metrics(evidence):
    groups = {}
    for case in evidence:
        key = (case["dataset"], case["task"], case["phase"])
        groups.setdefault(key, []).append(case)

    report = []
    for (dataset, task, phase), cases in sorted(groups.items()):
        counts = Counter(case["outcome"] for case in cases)
        tp, tn, fp, fn = (counts[k] for k in ("TP", "TN", "FP", "FN"))
        test_n = len(cases)
        risk_n = sum(case["expected_risk"] for case in cases)
        benign_n = test_n - risk_n
        scored_n = test_n - counts["U"]
        report.append({
            "dataset": dataset, "task": task, "phase": phase,
            "test_n": test_n, "scored_n": scored_n, "unscored_n": counts["U"],
            "risk_n": risk_n, "benign_n": benign_n, "correct": tp + tn,
            "tp": tp, "tn": tn, "fp": fp, "fn": fn,
            "accuracy": (tp + tn) / test_n,
            "risk_recall": tp / risk_n if risk_n else None,
            "benign_allow_rate": tn / benign_n if benign_n else None,
            "false_positive_rate": fp / (tn + fp) if tn + fp else None,
            "false_negative_rate": fn / (tp + fn) if tp + fn else None,
            "coverage": scored_n / test_n,
        })
    return report


def run_original_baseline(agents):
    reports, evidence = {}, {}
    for split, title in BASELINE_SPLITS.items():
        rows = read_split(split)
        print(f"Scoring original Laya on {len(rows):,} rows ({title}).", flush=True)
        evidence[split] = baseline_case_evidence(split, rows, score_split(agents, rows))
        reports[split] = baseline_metrics(evidence[split])
    return reports, evidence


def save_baseline_evidence(evidence):
    destination = OUTPUT / "original-laya-cases.jsonl"
    destination.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=destination.parent) as directory:
        staged = Path(directory) / destination.name
        with staged.open("w", encoding="utf-8") as stream:
            for cases in evidence.values():
                for case in cases:
                    stream.write(json.dumps(case) + "\n")
        staged.replace(destination)
    return destination


def show_case_rows(cases):
    import pandas as pd
    from IPython.display import display

    table = pd.DataFrame(cases, columns=list(CASE_COLUMNS))
    for field in ("expected_risk", "predicted_risk"):
        table[field] = table[field].map({0: "Benign", 1: "Risk"})
    table["probability"] = table["probability"].map(str, na_action="ignore")
    table = table.astype(object).where(table.notna(), "—")
    with pd.option_context("display.max_columns", None, "display.max_colwidth", None):
        display(table.rename(columns=CASE_COLUMNS))


def show_baseline_cases(split="acceptance", *, outcome=None, dataset=None, limit=20):
    cases = ORIGINAL_BASELINE_CASES[split]
    if outcome is not None:
        outcome = outcome.upper()
        cases = [case for case in cases if case["outcome"] == outcome]
    if dataset is not None:
        cases = [case for case in cases if case["dataset"] == dataset]
    print(f"{len(cases):,} matching cases in {BASELINE_SPLITS[split]}")
    show_case_rows(cases[:limit])


def inspect_baseline_case(split, case_id):
    case = next((case for case in ORIGINAL_BASELINE_CASES[split] if case["case_id"] == case_id), None)
    if case is None:
        raise KeyError(f"No baseline case {case_id!r} in {split}.")
    row = next(row for row in read_split(split) if row["id"] == case_id)
    return {**case, "question": QUESTIONS[case["task"]], "model_input": state_for(row)}


def show_original_baseline(reports, evidence):
    import pandas as pd
    from IPython.display import display

    print("Original Laya baseline")
    for split, title in BASELINE_SPLITS.items():
        print(title)
        table = pd.DataFrame(reports[split], columns=[*BASELINE_COLUMNS, "phase"])
        table["task"] = table["task"].map(TASK_LABELS) + " (" + table["phase"] + ")"
        table = table.drop(columns="phase")
        rates = list(BASELINE_RATE_COLUMNS)
        percentages = (table[rates].astype(float) * 100).round(2)
        table[rates] = percentages.astype(object).where(percentages.notna(), "—")
        with pd.option_context("display.max_columns", None):
            display(table.rename(columns=BASELINE_COLUMNS))

    acceptance = evidence["acceptance"]
    examples = [
        next((case for case in acceptance if case["outcome"] == outcome), None)
        for outcome in ("FP", "FN", "TP", "TN", "U")
    ]
    print(f"Frozen acceptance case examples (risk cutoff {BASELINE_CUTOFF:g})")
    show_case_rows([case for case in examples if case is not None])


MODEL_DIR = materialize()
with scoring_agents(MODEL_DIR, expected_sha256=BASE["files"]) as agents:
    TOKENIZER = agents[0].tok
    ORIGINAL_BASELINE, ORIGINAL_BASELINE_CASES = run_original_baseline(agents)
BASELINE_EVIDENCE_PATH = save_baseline_evidence(ORIGINAL_BASELINE_CASES)
show_original_baseline(ORIGINAL_BASELINE, ORIGINAL_BASELINE_CASES)
print(f"Saved all case evidence to {BASELINE_EVIDENCE_PATH}.")

## Review baseline cases

The original checkpoint predicts risk at `P(risk) ≥ 0.5`. TP, TN, FP and FN compare that
prediction with the dataset label. The expected action is also part of the dataset annotation.

- Acc uses all test cases. RR uses all risk cases and BAR uses all benign cases.
- FPR and FNR use the scored cases of the corresponding class. Truncated inputs and missing or
  invalid predictions count as unscored (`U`). A dash marks a rate with a zero denominator.

Filter either suite in a new code cell.

~~~python
show_baseline_cases("acceptance", outcome="FN", limit=10)
~~~

Pass a Case ID from the table to `inspect_baseline_case("acceptance", case_id)` to see the exact
model question and input alongside its verdict. The full case evidence is saved in
`laya-guardrails/original-laya-cases.jsonl` under the working directory.


### Define training preprocessing

Defines how the training cell tokenizes and prepares the train and validation splits. Run this cell before launching training.

In [ ]:
import time

import torch
from laya.common import QTYPES, build_sequence, render_options


def encode(rows, name):
    items = []
    started = time.time()
    for index, row in enumerate(rows, start=1):
        question = {"t": "noul", "ins": QUESTIONS[task_for(row)], "crit": None}
        ids, markers, options, usage = build_sequence(
            TOKENIZER,
            state_for(row),
            question,
            max_len=TRAINING["maxLen"],
            head_max_len=TRAINING["headMaxLen"],
            return_stats=True,
            return_truncation_stats=True,
        )
        if len(markers) != len(render_options(question)) or options["options_distinct"] != 2:
            raise ValueError(f"Could not encode both Noul options for {row['id']}.")
        # collate_items reads "label"; a "target" would have to carry one entry per option.
        items.append(
            {
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES["noul"],
                "label": row["label"],
                "truncated": usage["truncated"],
            }
        )
        if index % 10000 == 0:
            print(f"  {name}: {index:,}/{len(rows):,} in {time.time() - started:.0f}s", flush=True)
    truncated = sum(item["truncated"] for item in items)
    print(f"{name}: {len(items):,} items, {truncated:,} truncated", flush=True)
    return items


def prepare_training_items():
    items = {split: encode(read_split(split), split) for split in ("train", "validation")}
    items_path = WORK / "laya-items.pt"
    torch.save(items, items_path)
    print(f"items: {items_path} ({items_path.stat().st_size / 1e6:.0f} MB)")
    return items_path

## Training

| Setting | Value |
| --- | --- |
| Method | LoRA rank 8, alpha 16, dropout 0.05, all linear layers of the encoder; decision head, type embedding and scorer stay trainable |
| Learning rates | encoder 1e-4, head 5e-5, AdamW, weight decay 0.01, gradient clip 1.0 |
| Batch | 8 sequences per forward pass per GPU, effective batch 16 across 2 GPUs, 4,096-token cap per micro batch |
| Epochs | 1 (49,430 rows / batch 16 ≈ 3,090 updates) |
| Validation | every 250 updates over the whole 6,314-row split; the best checkpoint wins on accuracy, then fewest false negatives |
| Precision | fp16 with a gradient scaler (a T4 has no usable bf16) |

Artifacts land in `/kaggle/working/laya-guardrails/`:

| File | Content |
| --- | --- |
| `latest.pt` | LoRA and head parameters, optimizer, scaler and step at each validation checkpoint |
| `best.pt` | parameters of the checkpoint with the highest validation accuracy |
| `model/` | the final merged checkpoint, loadable with `laya.load` |
| `training.json` | receipt: dataset identity, recipe, validation result, file digests |

Set `RUN_TRAINING = True` in the `train` cell to launch `torchrun`. The `train-script` cell
writes the script to `/kaggle/working/train_laya.py`. Training reloads `latest.pt` when it
is present in the working directory.

### Save the checkpoint in Kaggle

In an interactive session, run through `upload-hf` and look for `Checkpoint ready`.
Then open **Save Version → Quick Save → Advanced Settings** and enable saving output
files. After the save finishes, open the version's **Output** tab and expand
`laya-guardrails/model/` to check the weights, config and tokenizer files.

For a background training run, set `RUN_TRAINING = True` before choosing **Save & Run All**
and check **Output** after the version completes. To also back up the model to Hugging
Face before evaluation, configure `HF_REPO_ID`, `RUN_HF_UPLOAD` and the `HF_TOKEN` secret.

### Reuse a saved checkpoint

Attach the saved version's Notebook Output as a Kaggle Input alongside the pinned
classification dataset. Keep `RUN_TRAINING = False` and run the notebook. Evaluation looks
under `/kaggle/input` for `laya-guardrails/model/` and loads the complete checkpoint there.
It uses the model in `/kaggle/working` after training or when no saved Output is mounted.
Set `TUNED_MODEL_DIR` in `config` to the exact model directory to choose a specific
checkpoint, then rerun `evaluate` and `fpr-budget`.

### Export FP16 weights

`EXPORT_PRECISION` in `config` defaults to `fp16` for new training runs. This stores
the merged weights in half precision. The checkpoint still loads with `laya.load`.
The temperature keeps its original precision.

For an existing checkpoint, attach its Notebook Output or set `TUNED_MODEL_DIR` in
`config`. Keep `RUN_TRAINING = False`, run `train-script`, then enable
`CONVERT_EXISTING_CHECKPOINT` in `export-fp16`. Save the converted `laya-guardrails/model-fp16/`
directory in your Kaggle version. This session uses it for Hugging Face upload, the model
comparison and the FPR budget evaluation. Rerun `evaluate` before `fpr-budget` if the
previous comparison used a different checkpoint.

### Export and evaluate INT8 ONNX

Attach a saved merged checkpoint and the classification dataset as Inputs. Set
`RUN_INT8_EXPORT = True` in `config` and keep `RUN_TRAINING = False`. Run the
notebook. `export-int8` uses the selected FP16 checkpoint for both backends. If
the source stores FP32, it saves an FP16 reference alongside the new export.

The graph and its tokenizer and configs are under
`laya-guardrails/int8-onnx/<run>/model/`. The graph uses Guard's dynamic System
One tensor schema and ONNX Runtime's CPU execution provider. Guard's offline
replay loader can use this directory and its `model.onnx`.

`int8-fpr-budget` uses the same `FPR_BUDGET`, `calibration_policy` partition and
`acceptance` suite as `fpr-budget`. Both backends run CPU inference with the same
questions and token budgets. The cell selects a threshold per task and phase for
each backend and compares acceptance RR and FPR on paired cases. Find
`summary.csv` and the two case JSONL files in the export run's `evaluation/` directory.

The same scored acceptance cases also produce the standard Evaluation table at
`BASELINE_CUTOFF` and `fixed-cutoff-acceptance.csv`. Use it to compare the
quantization effect at a shared operating point. The FPR budget report compares
the operating points selected separately on the calibration partition.


### Write the training script

Writes the two-GPU training loop and checkpoint export helpers to `/kaggle/working/train_laya.py`. Run it once per session before training or converting an existing checkpoint to FP16.

In [ ]:
%%writefile /kaggle/working/train_laya.py
# Joint LoRA and decision-head fine-tune for the Laya guardrails dataset.
import hashlib
import json
import os
import random
import shutil
import sys
from contextlib import nullcontext
from pathlib import Path
from tempfile import mkdtemp

import torch
import torch.distributed as dist
import torch.nn.functional as F
from safetensors import safe_open
from safetensors.torch import load_file, save_file
from torch.nn.parallel import DistributedDataParallel as DDP
from transformers import AutoTokenizer

from laya.common import build_model, collate_items

TENSORS = ("input_ids", "attention_mask", "marker_pos", "marker_mask", "qtype")


def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


EXPORT_DTYPES = {"fp16": torch.float16, "fp32": torch.float32}


def checkpoint_weights(tensors, precision):
    float_dtype = EXPORT_DTYPES[precision]
    weights = {}
    for name, value in tensors:
        # Keep the temperature and non-floating buffers at their stored precision.
        dtype = float_dtype if name != "temperature" and value.is_floating_point() else value.dtype
        tensor = value.detach().to(device="cpu", dtype=dtype).contiguous()
        if dtype == torch.float16 and not torch.isfinite(tensor).all().item():
            raise ValueError(f"Non-finite values after casting {name} to FP16.")
        weights[name] = tensor
    return weights


def write_checkpoint(tensors, cfg, directory, source, required_files, precision):
    # Stage the layout laya.load expects before replacing the current model directory.
    if directory.exists() and not directory.is_dir():
        raise NotADirectoryError(directory)
    directory.parent.mkdir(parents=True, exist_ok=True)
    staging = Path(mkdtemp(prefix=".model-export-", dir=directory.parent))
    candidate = staging / "model"
    previous = staging / "previous"
    published = False
    try:
        candidate.mkdir()
        (candidate / "rl_agent_config.json").write_text(json.dumps(cfg, indent=2))
        save_file(
            checkpoint_weights(tensors, precision),
            str(candidate / "model.safetensors"),
            metadata={"format": "pt", "storage_precision": precision},
        )
        for name in ("tokenizer", "encoder"):
            for path in sorted((source / name).rglob("*")):
                if path.is_file():
                    target = candidate / name / path.relative_to(source / name)
                    target.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copyfile(path, target)

        incomplete = [
            name for name in required_files
            if not (candidate / name).is_file() or (candidate / name).stat().st_size == 0
        ]
        if incomplete:
            raise FileNotFoundError(f"Missing or empty model files in staged export: {', '.join(incomplete)}.")

        if directory.exists():
            directory.replace(previous)
        candidate.replace(directory)
        published = True
    except BaseException:
        if previous.exists() and not directory.exists():
            previous.replace(directory)
        raise
    finally:
        if published or not previous.exists():
            shutil.rmtree(staging)


def export_fp16(source, destination, required_files):
    if source.resolve() == destination.resolve():
        raise ValueError("Use a different directory for the FP16 export.")
    original_weights = source / "model.safetensors"
    original_size = original_weights.stat().st_size
    cfg = json.loads((source / "rl_agent_config.json").read_text())
    with safe_open(str(original_weights), framework="pt", device="cpu") as original:
        tensors = ((name, original.get_tensor(name)) for name in original.keys())
        write_checkpoint(tensors, cfg, destination, source, required_files, "fp16")
    weight_size = (destination / "model.safetensors").stat().st_size
    print(f"Saved FP16 checkpoint at {destination}. Weights {original_size:,} → {weight_size:,} bytes.")


def save(path, value):
    staged = path.with_suffix(path.suffix + ".tmp")
    torch.save(value, staged)
    staged.replace(path)


def restore(params, saved):
    if set(params) != set(saved):
        raise ValueError("Checkpoint parameter names differ from the configured model.")
    with torch.no_grad():
        for name, param in params.items():
            param.copy_(saved[name])


def epoch_batches(items, rank, workers, microbatch, budget, epoch):
    # Length-sorted so an 8,192-token document travels alone, then shuffled as a batch list.
    order = sorted(range(rank, len(items), workers), key=lambda i: -len(items[i]["ids"]))
    batches, current, tokens = [], [], 0
    for index in order:
        width = len(items[index]["ids"])
        if current and (len(current) >= microbatch or tokens + width > budget):
            batches.append(current)
            current, tokens = [], 0
        current.append(index)
        tokens += width
    if current:
        batches.append(current)
    random.Random(f"{epoch}-{rank}").shuffle(batches)
    return batches


def collect(values, device):
    return (
        {key: values[key].to(device) for key in TENSORS},
        values["label"].to(device),
    )


def validation_shards(items, workers, width):
    # Keep each batch intact. Estimate attention work from the padded sequence length.
    costs = []
    for start in range(0, len(items), width):
        chunk = items[start : start + width]
        padded_length = max(len(item["ids"]) for item in chunk)
        costs.append((len(chunk) * padded_length**2, start))

    shards = [[] for _ in range(workers)]
    work = [0] * workers
    for cost, start in sorted(costs, reverse=True):
        rank = min(range(workers), key=work.__getitem__)
        shards[rank].append(start)
        work[rank] += cost
    return shards


def validate(model, items, tok, device, starts, width):
    model.eval()
    counts = torch.zeros(3, dtype=torch.long, device=device)
    with torch.no_grad():
        for start in starts:
            chunk = items[start : start + width]
            batch, target = collect(collate_items([chunk], tok.pad_token_id), device)
            with torch.autocast("cuda", dtype=torch.float16):
                logits, _ = model(**batch)
            predicted = (logits[:, 1] >= logits[:, 0]).long()
            counts[0] += int((predicted == target).sum())
            counts[1] += len(target)
            counts[2] += int(((predicted == 0) & (target == 1)).sum())
    dist.all_reduce(counts)
    model.train()
    return {
        "scored": int(counts[1]),
        "correct": int(counts[0]),
        "falseNegatives": int(counts[2]),
        "accuracy": float(counts[0]) / max(1, int(counts[1])),
    }


def main():
    from peft import LoraConfig, get_peft_model

    spec = json.loads(Path(sys.argv[1]).read_text())
    precision = spec.get("exportPrecision", "fp32")
    if precision not in EXPORT_DTYPES:
        raise ValueError("Choose fp16 or fp32 for EXPORT_PRECISION.")
    settings, optimizer_settings = spec["training"], spec["training"]["optimizer"]
    lora_settings = settings["lora"]
    output = Path(spec["output"])
    output.mkdir(parents=True, exist_ok=True)

    dist.init_process_group("nccl")
    rank, workers = dist.get_rank(), dist.get_world_size()
    local = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local)
    device = torch.device("cuda", local)
    torch.manual_seed(settings["seed"] + rank)
    torch.cuda.manual_seed(settings["seed"] + rank)

    model_dir = Path(spec["modelDir"])
    cfg = json.loads((model_dir / "rl_agent_config.json").read_text())
    cfg.update(
        max_len=settings["maxLen"],
        head_max_len=settings["headMaxLen"],
        gradient_checkpointing=settings["gradientCheckpointing"],
    )
    tok = AutoTokenizer.from_pretrained(model_dir / "tokenizer")
    model = build_model(cfg, encoder_dir=str(model_dir / "encoder"), pretrained=False)
    model.load_state_dict(load_file(model_dir / "model.safetensors"), strict=True)

    model.requires_grad_(False)
    model.encoder = get_peft_model(
        model.encoder,
        LoraConfig(
            r=lora_settings["rank"],
            lora_alpha=lora_settings["alpha"],
            lora_dropout=lora_settings["dropout"],
            target_modules=lora_settings["targetModules"],
            bias="none",
        ),
    )
    for module in (model.head, model.type_emb, model.scorer):
        if module is not None:
            module.requires_grad_(True)
    model.act_head.requires_grad_(False)  # Noul supervision supplies no action target.
    if settings["gradientCheckpointing"]:
        model.encoder.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
        model.head_checkpointing = True
    model.to(device).train()
    ddp = DDP(model, device_ids=[local], broadcast_buffers=False, gradient_as_bucket_view=True)

    params = {name: p for name, p in model.named_parameters() if p.requires_grad}
    encoder = [p for name, p in params.items() if name.startswith("encoder.")]
    head = [p for name, p in params.items() if not name.startswith("encoder.")]
    optimizer = torch.optim.AdamW(
        [
            {"params": encoder, "lr": optimizer_settings["encoderLR"]},
            {"params": head, "lr": optimizer_settings["headLR"]},
        ],
        weight_decay=optimizer_settings["weightDecay"],
    )
    scaler = torch.amp.GradScaler("cuda", enabled=True)
    micro = settings["microBatch"]
    if micro * workers > settings["effectiveBatch"]:
        raise ValueError("microBatch times workers exceeds effectiveBatch.")
    accumulation = max(1, settings["effectiveBatch"] // (micro * workers))

    items = torch.load(spec["items"], weights_only=False)
    train_items, val_items = items["train"], items["validation"]
    val_starts = validation_shards(val_items, workers, micro)[rank]
    latest, best_path = output / "latest.pt", output / "best.pt"
    step, best = 0, None
    resumed = spec.get("resume")
    if resumed and Path(resumed).is_file():
        state = torch.load(resumed, map_location="cpu", weights_only=True)
        restore(params, state["params"])
        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        step, best = state["step"], state["best"]
        if rank == 0:
            print(f"resumed from {resumed} at step {step}", flush=True)

    skip = step
    for epoch in range(settings["epochs"]):
        batches = epoch_batches(
            train_items, rank, workers, micro, settings["tokenBudget"], epoch
        )
        # Both ranks must run the same number of update steps or DDP hangs on the all-reduce.
        count = torch.tensor([len(batches)], device=device)
        dist.all_reduce(count, op=dist.ReduceOp.MIN)
        batches = batches[: int(count)]
        if rank == 0:
            print(
                f"epoch {epoch + 1}/{settings['epochs']}: {len(train_items):,} rows over "
                f"{len(batches)} batches per rank, resuming after {skip} updates",
                flush=True,
            )
        losses, updates = [], 0
        for start in range(0, len(batches), accumulation):
            if step < skip:
                step += 1
                continue
            group = batches[start : start + accumulation]
            optimizer.zero_grad(set_to_none=True)
            total = torch.zeros((), device=device)
            for index, batch_indices in enumerate(group):
                synced = index + 1 == len(group)
                with (nullcontext() if synced else ddp.no_sync()):
                    batch, target = collect(
                        collate_items([[train_items[i] for i in batch_indices]], tok.pad_token_id),
                        device,
                    )
                    with torch.autocast("cuda", dtype=torch.float16):
                        logits, _ = ddp(**batch)
                        loss = F.cross_entropy(logits.float(), target, reduction="sum")
                    scaler.scale(loss * (workers / settings["effectiveBatch"])).backward()
                total += loss.detach()
            dist.all_reduce(total)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(list(params.values()), optimizer_settings["maxGradNorm"])
            scaler.step(optimizer)
            scaler.update()
            step += 1
            updates += 1
            losses.append(float(total) / settings["effectiveBatch"])
            if step % settings["checkpointSteps"] == 0 or start + accumulation >= len(batches):
                if rank == 0:
                    print(
                        f"Validating step {step} on {len(val_items):,} rows across {workers} GPU ranks.",
                        flush=True,
                    )
                metrics = validate(model, val_items, tok, device, val_starts, micro)
                score = (metrics["correct"], -metrics["falseNegatives"])
                if best is None or score > (best["correct"], -best["falseNegatives"]):
                    best = metrics
                    if rank == 0:
                        save(best_path, {"params": {n: p.detach().cpu() for n, p in params.items()}})
                if rank == 0:
                    save(
                        latest,
                        {
                            "params": {n: p.detach().cpu() for n, p in params.items()},
                            "optimizer": optimizer.state_dict(),
                            "scaler": scaler.state_dict(),
                            "step": step,
                            "best": best,
                        },
                    )
                    recent = sum(losses[-settings["checkpointSteps"] :]) / max(
                        1, len(losses[-settings["checkpointSteps"] :])
                    )
                    print(
                        f"epoch {epoch + 1}/{settings['epochs']} step {step} "
                        f"loss {recent:.4f} accuracy {metrics['accuracy']:.4f} "
                        f"({metrics['correct']}/{metrics['scored']})",
                        flush=True,
                    )
    dist.barrier()
    if rank == 0:
        print("merging the adapter and saving the final checkpoint", flush=True)
        if best_path.is_file():
            restore(params, torch.load(best_path, map_location="cpu", weights_only=True)["params"])
        merged = ddp.module
        merged.encoder = merged.encoder.merge_and_unload(safe_merge=True)
        out_cfg = dict(cfg, max_len=settings["maxLen"], head_max_len=settings["headMaxLen"])
        out_cfg.update(fine_tuned=True, model_name="laya-guardrails")
        exported_model = output / "model"
        write_checkpoint(
            merged.state_dict().items(), out_cfg, exported_model, model_dir, spec["modelFiles"], precision
        )
        weights_path = exported_model / "model.safetensors"
        files = {
            path.relative_to(output).as_posix(): sha256(path)
            for path in sorted(exported_model.rglob("*"))
            if path.is_file()
        }
        receipt = {
            "datasetId": spec["dataset"]["datasetId"],
            "datasetVersion": spec["dataset"]["version"],
            "datasetManifestSha256": spec["dataset"]["sha256"],
            "baseModel": spec["base"],
            "training": settings,
            "exportPrecision": precision,
            "modelWeightBytes": weights_path.stat().st_size,
            "updates": step,
            "validation": best,
            "checkpointSha256": sha256(best_path) if best_path.is_file() else None,
            "latestCheckpointSha256": sha256(latest) if latest.is_file() else None,
            "files": files,
        }
        receipt_path = output / "training.json"
        staged_receipt = receipt_path.with_suffix(".json.tmp")
        staged_receipt.write_text(json.dumps(receipt, indent=2, sort_keys=True))
        staged_receipt.replace(receipt_path)
        print(f"Saved {precision.upper()} weights at {weights_path} ({weights_path.stat().st_size:,} bytes).", flush=True)
        print(f"Training receipt saved at {receipt_path}", flush=True)
    dist.destroy_process_group()


if __name__ == "__main__":
    main()


### Run fine-tuning

Set `RUN_TRAINING = True` to fine-tune on two T4 GPUs. The merged checkpoint is written to `laya-guardrails/model/`. Leave the flag off when evaluating a saved checkpoint.

In [ ]:
RUN_TRAINING = False  # the one long job in this Notebook; set it when you mean it

import json
import subprocess
import sys

if RUN_TRAINING:
    if EXPORT_PRECISION not in ("fp16", "fp32"):
        raise ValueError("Choose fp16 or fp32 for EXPORT_PRECISION.")
    SCRIPT = WORK / "train_laya.py"
    if not SCRIPT.is_file():
        raise FileNotFoundError(f"{SCRIPT} is missing; run the cell above, which writes it.")
    ITEMS_PATH = prepare_training_items()
    TRAIN_SPEC = WORK / "laya-guardrails.train-spec.json"
    TRAIN_SPEC.write_text(
        json.dumps(
            {
                "dataset": DATASET,
                "base": {"id": BASE["id"], "revision": BASE["revision"]},
                "training": TRAINING,
                "modelDir": str(MODEL_DIR),
                "modelFiles": list(BASE["files"]),
                "exportPrecision": EXPORT_PRECISION,
                "items": str(ITEMS_PATH),
                "output": str(OUTPUT),
                "resume": str(OUTPUT / "latest.pt"),
            },
            indent=2,
        )
    )
    command = [
        sys.executable,
        "-m",
        "torch.distributed.run",
        "--standalone",
        f"--nproc-per-node={TRAINING['workers']}",
        str(SCRIPT),
        str(TRAIN_SPEC),
    ]
    print(" ".join(command))
    subprocess.run(command, check=True)
    FP16_EXPORT_DIR = None
else:
    print("Training skipped. Evaluation can use a saved merged checkpoint.")


### Convert an existing checkpoint to FP16

Set `CONVERT_EXISTING_CHECKPOINT = True` to convert a merged FP32 checkpoint. Run after writing the training script. The converted model is saved under `laya-guardrails/model-fp16/`.

In [ ]:
from runpy import run_path

CONVERT_EXISTING_CHECKPOINT = False  # Enable to export an already merged checkpoint as FP16.

FP16_EXPORT_DIR = None
if CONVERT_EXISTING_CHECKPOINT:
    source_checkpoint = resolve_tuned_model_dir()
    if source_checkpoint is None:
        raise FileNotFoundError("Attach a saved checkpoint or set TUNED_MODEL_DIR in config.")
    base_dir = globals().get("MODEL_DIR")
    if base_dir is not None and source_checkpoint.resolve() == Path(base_dir).resolve():
        raise ValueError("Select the merged fine-tuned checkpoint for FP16 export.")

    destination = OUTPUT / "model-fp16"
    if source_checkpoint.resolve() == destination.resolve():
        print(f"Reusing the existing export at {destination}.")
    else:
        script = WORK / "train_laya.py"
        if not script.is_file():
            raise FileNotFoundError(f"{script} is missing. Run train-script first.")
        run_path(str(script))["export_fp16"](source_checkpoint, destination, BASE["files"])
    model_export_sizes(destination)
    FP16_EXPORT_DIR = destination

### Export the INT8 ONNX model

Exports and smoke-tests a Guard-compatible ONNX model from the selected checkpoint. Enable `RUN_INT8_EXPORT` in the configuration cell before running this cell.

In [ ]:
import json
import math
import shutil
import threading
from contextlib import contextmanager
from datetime import datetime, timezone
from runpy import run_path
from tempfile import TemporaryDirectory

import laya
import torch


GUARD_ONNX_OPSET = 17
INT8_CPU_THREADS = 2
GUARD_ONNX_INPUTS = {
    "input_ids": ("tensor(int64)", 2),
    "attention_mask": ("tensor(int64)", 2),
    "marker_pos": ("tensor(int64)", 2),
    "marker_mask": ("tensor(bool)", 2),
    "qtype": ("tensor(int64)", 1),
}
GUARD_ONNX_OUTPUTS = {"logits": ("tensor(float)", 2), "act_logits": ("tensor(float)", 2)}


class GuardOnnxExport(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids, attention_mask, marker_pos, marker_mask, qtype):
        # The masked slot keeps the model's TopK valid for a single-option question.
        padded_pos = torch.cat((marker_pos, torch.zeros_like(marker_pos[:, :1])), dim=1)
        padded_mask = torch.cat((marker_mask, torch.zeros_like(marker_mask[:, :1])), dim=1)
        logits, act_logits = self.model(input_ids, attention_mask, padded_pos, padded_mask, qtype)
        return logits[:, :-1], act_logits


def guard_onnx_example(batch, tokens, options):
    input_ids = torch.arange(1, tokens + 1, dtype=torch.int64).expand(batch, -1).clone()
    attention_mask = torch.ones_like(input_ids)
    attention_mask[:, -2:] = 0
    marker_pos = torch.arange(1, options + 1, dtype=torch.int64).expand(batch, -1).clone()
    marker_mask = torch.ones_like(marker_pos, dtype=torch.bool)
    if batch > 1 and options > 1:
        marker_pos[-1, -1] = -1
        marker_mask[-1, -1] = False
    qtype = torch.zeros(batch, dtype=torch.int64)
    return input_ids, attention_mask, marker_pos, marker_mask, qtype


def checkpoint_stores_fp16(directory):
    from safetensors import safe_open

    with safe_open(str(directory / "model.safetensors"), framework="pt", device="cpu") as weights:
        dtypes = {weights.get_slice(name).get_dtype() for name in weights.keys() if name != "temperature"}
    return "F16" in dtypes and not any(kind != "F16" and kind.startswith(("F", "BF")) for kind in dtypes)


@contextmanager
def fp16_cpu_agents(directory):
    threads = torch.get_num_threads()
    torch.set_num_threads(INT8_CPU_THREADS)
    agents = []
    try:
        agent = laya.load(str(directory), device="cpu", fast=False, compile=False)
        agent.model.float().eval()
        agent.amp_enabled = False
        calibration = directory / "calibration.json"
        if calibration.is_file():
            agent.load_calibration(str(calibration))
        agents.append(agent)
        yield agents
    finally:
        agents.clear()
        torch.set_num_threads(threads)


def export_guard_onnx_graph(directory, graph_path):
    dynamic_axes = {name: {0: "questions"} for name in (*GUARD_ONNX_INPUTS, *GUARD_ONNX_OUTPUTS)}
    for name in ("input_ids", "attention_mask"):
        dynamic_axes[name][1] = "tokens"
    for name in ("marker_pos", "marker_mask", "logits"):
        dynamic_axes[name][1] = "options"

    with fp16_cpu_agents(directory) as agents:
        model = GuardOnnxExport(agents[0].model).eval()
        fastpath = torch.backends.mha.get_fastpath_enabled()
        try:
            torch.backends.mha.set_fastpath_enabled(False)
            with torch.no_grad():
                torch.onnx.export(
                    model, guard_onnx_example(2, 16, 2), str(graph_path),
                    input_names=list(GUARD_ONNX_INPUTS), output_names=list(GUARD_ONNX_OUTPUTS),
                    dynamic_axes=dynamic_axes, opset_version=GUARD_ONNX_OPSET,
                    dynamo=False, external_data=False,
                )
        finally:
            torch.backends.mha.set_fastpath_enabled(fastpath)


def verify_guard_onnx_schema(session):
    for nodes, expected in (
        (session.get_inputs(), GUARD_ONNX_INPUTS), (session.get_outputs(), GUARD_ONNX_OUTPUTS),
    ):
        if {node.name: (node.type, len(node.shape)) for node in nodes} != expected:
            raise ValueError("ONNX tensor schema differs from the Guard runtime contract.")
        if any(isinstance(node.shape[0], int) for node in nodes):
            raise ValueError("ONNX tensors must accept a dynamic question batch.")
        for node in nodes:
            if node.name in {"input_ids", "attention_mask", "marker_pos", "marker_mask", "logits"}:
                if isinstance(node.shape[1], int):
                    raise ValueError(f"ONNX tensor {node.name} has a fixed token or option length.")


@contextmanager
def int8_cpu_agents(directory):
    import onnxruntime as ort
    from laya.calibrate import apply_calibration_payload
    from laya.onnx_agent import ONNXAgent
    from transformers import AutoTokenizer

    # Match the Guard loader's CPU session and bind Laya's System One request handling.
    options = ort.SessionOptions()
    options.intra_op_num_threads = INT8_CPU_THREADS
    options.inter_op_num_threads = 1
    options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    session = ort.InferenceSession(
        str(directory / "model.onnx"), sess_options=options, providers=["CPUExecutionProvider"]
    )
    verify_guard_onnx_schema(session)

    agent = ONNXAgent.__new__(ONNXAgent)
    agent.model_id = str(directory)
    agent.device = torch.device("cpu")
    agent.cfg = json.loads((directory / "rl_agent_config.json").read_bytes())
    agent.tok = AutoTokenizer.from_pretrained(str(directory / "tokenizer"), local_files_only=True)
    agent.session = session
    agent.hooks = []
    agent._hooks_mutex = threading.Lock()
    agent.lang_temperatures = {}
    calibration = directory / "calibration.json"
    if calibration.is_file():
        agent.load_calibration(str(calibration))
    else:
        apply_calibration_payload(agent, {
            "temperature": agent.cfg.get("temperature", [1, 1, 1]),
            "temperature_by_options": agent.cfg.get("temperature_by_options", {}),
        })

    agents = [agent]
    try:
        yield agents
    finally:
        agents.clear()


def smoke_test_guard_onnx(directory):
    import numpy as np

    with int8_cpu_agents(directory) as agents:
        agent = agents[0]
        for batch, tokens, options in ((1, 13, 1), (2, 57, 4), (1, RUNTIME["maxLen"], 2)):
            inputs = {
                name: value.numpy()
                for name, value in zip(GUARD_ONNX_INPUTS, guard_onnx_example(batch, tokens, options), strict=True)
            }
            logits, actions = agent.session.run(list(GUARD_ONNX_OUTPUTS), inputs)
            if logits.shape != (batch, options) or actions.shape[0] != batch:
                raise RuntimeError("INT8 ONNX returned unexpected output shapes.")
            if not np.isfinite(logits).all() or not np.isfinite(actions).all():
                raise RuntimeError("INT8 ONNX returned non-finite output.")
        payload = agent.system_one(
            "The meeting is at noon.",
            {"decision": {"type": "noul", "instructions": "Does the text mention a meeting?"}},
            lang=RUNTIME["language"], max_len=RUNTIME["maxLen"], head_max_len=RUNTIME["headMaxLen"],
        )
        probability = payload["answers"]["decision"]["noul"]
        if not math.isfinite(probability) or not 0 <= probability <= 1:
            raise RuntimeError("The INT8 checkpoint did not return a valid System One probability.")


def export_int8_checkpoint(source):
    import onnx
    import onnxruntime as ort
    from onnxruntime.quantization import QuantType, quantize_dynamic

    source = Path(source)
    model_export_sizes(source)
    output_root = OUTPUT / "int8-onnx"
    output_root.mkdir(parents=True, exist_ok=True)
    destination = output_root / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    with TemporaryDirectory(dir=output_root) as temporary:
        scratch = Path(temporary)
        staged = scratch / "result"
        model_dir = staged / "model"
        model_dir.mkdir(parents=True)
        fp16 = source
        if not checkpoint_stores_fp16(source):
            script = WORK / "train_laya.py"
            if not script.is_file():
                raise FileNotFoundError(f"{script} is missing. Run train-script first.")
            fp16 = staged / "fp16-reference"
            run_path(str(script))["export_fp16"](source, fp16, BASE["files"])
            if (source / "calibration.json").is_file():
                shutil.copyfile(source / "calibration.json", fp16 / "calibration.json")

        print(f"Exporting the FP16 checkpoint through a float32 ONNX graph from {fp16}", flush=True)
        fp32_graph = scratch / "fp32.onnx"
        export_guard_onnx_graph(fp16, fp32_graph)
        int8_graph = model_dir / "model.onnx"
        print("Quantizing constant MatMul, Gemm and Gather weights with ONNX Runtime.", flush=True)
        quantize_dynamic(
            str(fp32_graph), str(int8_graph), per_channel=True, weight_type=QuantType.QInt8,
            op_types_to_quantize=["MatMul", "Gemm", "Gather"],
            extra_options={"MatMulConstBOnly": True}, use_external_data_format=False,
        )
        fp32_graph.unlink()
        graph = onnx.load(str(int8_graph), load_external_data=False)
        opset = next((item.version for item in graph.opset_import if item.domain in ("", "ai.onnx")), None)
        matmul_count = sum(node.op_type == "MatMulInteger" for node in graph.graph.node)
        int8_count = sum(tensor.data_type == onnx.TensorProto.INT8 for tensor in graph.graph.initializer)
        if opset != GUARD_ONNX_OPSET or not matmul_count or not int8_count:
            raise ValueError("Expected an opset 17 graph with INT8 weights and integer MatMul operators.")
        if any(tensor.data_location == onnx.TensorProto.EXTERNAL for tensor in graph.graph.initializer):
            raise ValueError("The INT8 graph must store its weights in model.onnx.")
        del graph

        assets = [name for name in BASE["files"] if name != "model.safetensors"]
        if (fp16 / "calibration.json").is_file():
            assets.append("calibration.json")
        for name in assets:
            target = model_dir / name
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(fp16 / name, target)

        smoke_test_guard_onnx(model_dir)
        reference = source if fp16 == source else destination / "fp16-reference"
        fp16_weights = fp16 / "model.safetensors"
        receipt = {
            "sourceCheckpoint": str(source),
            "fp16Reference": str(reference),
            "fp16WeightBytes": fp16_weights.stat().st_size,
            "fp16WeightSha256": sha256(fp16_weights),
            "onnxBytes": int8_graph.stat().st_size,
            "onnxOpset": opset,
            "onnxRuntimeVersion": ort.__version__,
            "quantization": "dynamic-int8-per-channel",
            "integerMatmuls": matmul_count,
            "int8Initializers": int8_count,
            "files": {name: sha256(model_dir / name) for name in ("model.onnx", *assets)},
        }
        (staged / "onnx-export.json").write_text(json.dumps(receipt, indent=2, sort_keys=True))
        staged.rename(destination)

    print(f"Saved {destination / 'model/model.onnx'} ({receipt['onnxBytes']:,} bytes).")
    return reference, destination / "model"


INT8_FP16_DIR = INT8_MODEL_DIR = None
if RUN_INT8_EXPORT:
    source_checkpoint = resolve_tuned_model_dir()
    if source_checkpoint is None:
        raise FileNotFoundError("Attach a saved merged Laya checkpoint or set TUNED_MODEL_DIR in config.")
    base_dir = globals().get("MODEL_DIR")
    if base_dir is not None and source_checkpoint.resolve() == Path(base_dir).resolve():
        raise ValueError("Select the merged fine-tuned checkpoint for INT8 export.")
    INT8_FP16_DIR, INT8_MODEL_DIR = export_int8_checkpoint(source_checkpoint)

### Verify the export and optionally upload it

Lists the files in the selected checkpoint. To upload, set `RUN_HF_UPLOAD = True`, enter `HF_REPO_ID`, and add `HF_TOKEN` in Kaggle Secrets.

In [ ]:
import os
from pathlib import Path


RUN_HF_UPLOAD = False
HF_REPO_ID = ""  # Hugging Face owner/model-name
HF_PRIVATE = True


def read_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError:
        token = os.environ.get("HF_TOKEN", "")
    else:
        try:
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            raise RuntimeError("Add and enable HF_TOKEN in Kaggle Add-ons → Secrets.") from None
    if not token or not token.strip():
        raise RuntimeError("Set HF_TOKEN in Kaggle Secrets or in the environment for a local run.")
    return token.strip()


def upload_hf_checkpoint(model_dir, repo_id, *, private):
    from huggingface_hub import HfApi
    from huggingface_hub.utils import validate_repo_id

    repo_id = repo_id.strip()
    if repo_id.count("/") != 1:
        raise ValueError("Set HF_REPO_ID to the target owner/model-name.")
    validate_repo_id(repo_id)

    model_dir = Path(model_dir)
    files = list(model_export_sizes(model_dir))

    api = HfApi(token=read_hf_token())
    api.create_repo(repo_id, repo_type="model", private=private, exist_ok=True)
    repo = api.repo_info(repo_id, repo_type="model")
    if private and not repo.private:
        raise ValueError(f"{repo_id} is public. Select a private repository or set HF_PRIVATE = False.")

    commit = api.upload_folder(
        repo_id=repo_id,
        repo_type="model",
        folder_path=model_dir,
        allow_patterns=files,
        parent_commit=repo.sha,
        commit_message=f"Upload Laya checkpoint from notebook revision {REVISION}",
    )
    return {
        "repoId": repo_id,
        "commit": commit.oid,
        "commitUrl": commit.commit_url,
        "private": repo.private,
    }


HF_UPLOAD_RECEIPT = None
export_dir = FP16_EXPORT_DIR or OUTPUT / "model"
if FP16_EXPORT_DIR is not None or RUN_HF_UPLOAD or globals().get("RUN_TRAINING", False):
    export_sizes = model_export_sizes(export_dir)
    print(f"Checkpoint ready at {export_dir}")
    for name, size in export_sizes.items():
        print(f"  {name} ({size:,} bytes)")
    print("In a Kaggle editor session, use Save Version → Quick Save and enable saving output files.")
    if RUN_HF_UPLOAD:
        HF_UPLOAD_RECEIPT = upload_hf_checkpoint(export_dir, HF_REPO_ID, private=HF_PRIVATE)
        print(f"Uploaded {HF_UPLOAD_RECEIPT['repoId']} at {HF_UPLOAD_RECEIPT['commitUrl']}")
else:
    print("Set HF_REPO_ID and RUN_HF_UPLOAD = True here to upload the exported model.")

### Compare the original and fine-tuned models

Scores the selected split at the standard 0.5 cutoff and shows TP, TN, FP, FN, and rates by dataset. Run after training or attaching the saved checkpoint to see both models side by side.

In [ ]:
from html import escape

import pandas as pd
from IPython.display import HTML, display


EVALUATION_TABLE_STYLE = """
<style>
.laya-evaluation-metrics { border-collapse: collapse; font-size: 14px; }
.laya-evaluation-metrics th, .laya-evaluation-metrics td {
    border: 0; padding: 6px 5px; text-align: right; white-space: nowrap;
}
.laya-evaluation-metrics th:nth-child(-n+2), .laya-evaluation-metrics td:nth-child(-n+2) {
    text-align: left;
}
.laya-evaluation-metrics thead tr { border-bottom: 1px solid rgba(127, 127, 127, .45); }
.laya-evaluation-metrics tbody tr:nth-child(even) { background: rgba(127, 127, 127, .09); }
</style>
"""


def metric_value(field, value):
    if value is None:
        return "—"
    if field in BASELINE_RATE_COLUMNS:
        return f"{value * 100:.2f}"
    return f"{value:,}"


def show_evaluation_comparison(results, *, labels=("Orig", "Tuned"), direction="Original → Post-trained"):
    original = results["base"]["by_dataset"]
    candidate = results.get("post-trained")
    group_fields = ("dataset", "task", "phase")
    tuned_by_group = {}
    if candidate is not None:
        tuned_by_group = {
            tuple(row[field] for field in group_fields): row
            for row in candidate["by_dataset"]
        }

    columns = {}
    for field, caption in BASELINE_COLUMNS.items():
        if field == "accuracy" and candidate is not None:
            columns.update({
                "original_accuracy": f"{labels[0]} Acc (%)",
                "tuned_accuracy": f"{labels[1]} Acc (%)",
                "accuracy_delta": "Δ Acc (pp)",
            })
        else:
            columns[field] = caption

    rows = []
    for before in original:
        task = TASK_LABELS[before["task"]]
        row = {"dataset": escape(before["dataset"]), "task": escape(f"{task} ({before['phase']})")}
        after = tuned_by_group[tuple(before[field] for field in group_fields)] if candidate is not None else None
        for field in BASELINE_COLUMNS:
            if field in row:
                continue
            first = metric_value(field, before[field])
            if after is None:
                row[field] = first
                continue
            second = metric_value(field, after[field])
            if field == "accuracy":
                delta = 100 * (after[field] - before[field])
                change = f"{delta:+.2f}" if delta else "0.00"
                if delta < 0:
                    second = f"<strong>{second}</strong>"
                    change = f"<strong>{change}</strong>"
                row.update(original_accuracy=first, tuned_accuracy=second, accuracy_delta=change)
            else:
                row[field] = first if before[field] == after[field] else f"{first} → {second}"
        rows.append(row)

    if candidate is not None:
        print(f"Other metrics read {direction}. A single value is shared.")
    table = pd.DataFrame(rows, columns=list(columns)).rename(columns=columns)
    html = table.to_html(index=False, escape=False, border=0, classes="laya-evaluation-metrics")
    display(HTML(EVALUATION_TABLE_STYLE + '<div style="max-width:100%; overflow-x:auto">' + html + "</div>"))


def run_evaluation():
    split = EVALUATION["split"]
    title = BASELINE_SPLITS.get(split, split)
    rows = read_split(split)
    if EVALUATION["limit"]:
        rows = rows[: EVALUATION["limit"]]

    calibration_rows = (
        read_split("calibration_fit")[: EVALUATION["calibrationSample"]]
        if EVALUATION["calibrate"] else []
    )

    models = {"base": MODEL_DIR}
    candidate = resolve_tuned_model_dir()
    if candidate is not None:
        if candidate.resolve() == Path(MODEL_DIR).resolve():
            raise ValueError("Original and tuned checkpoint directories must differ.")
        models["post-trained"] = candidate
        print(f"Using post-trained checkpoint at {candidate}")
    else:
        print("Showing Original only. Attach a saved Kaggle Notebook Output containing laya-guardrails/model/.")

    labels = {"base": "Original", "post-trained": "Post-trained"}
    results, evidence = {}, {}

    for name, directory in models.items():
        label = labels[name]
        print(f"Scoring {label} on {len(rows):,} rows ({title}).", flush=True)

        with scoring_agents(directory) as agents:
            scored = score_split(agents, rows)
            entry = {"split": split, "modelDir": str(directory), "summary": summarize(scored)}

            if calibration_rows:
                print(f"Fitting temperature for {label} on {len(calibration_rows):,} rows.", flush=True)
                sample = score_split(agents, calibration_rows)
                temperature = fit_temperature(sample["probability"], sample["label"])
                if temperature is not None:
                    entry["calibrated"] = summarize(scored, temperature)

        cases = baseline_case_evidence(split, rows, scored)
        metrics = baseline_metrics(cases)
        entry["by_dataset"] = metrics
        results[name] = entry
        evidence[name] = cases

    print(f"{title} · {len(rows):,} cases per model · risk cutoff {BASELINE_CUTOFF:g}")
    show_evaluation_comparison(results)

    return results, evidence


EVALUATION_CASE_COMPARISON = EVALUATION_CASE_EVIDENCE_PATH = None
EVALUATED, EVALUATION_CASES = run_evaluation()

### Inspect fixed and new errors

Joins predictions by `case_id` and saves both scores and decisions to JSONL. Run after the model comparison. Filter by `change` to find fixed or new FP and FN cases.

In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
from IPython.display import display


CASE_CONTEXT_FIELDS = (
    "split", "case_id", "dataset", "source", "category", "task", "phase",
    "expected_risk", "expected_action", "cutoff",
)
CASE_PREDICTION_FIELDS = (
    "probability", "predicted_risk", "outcome", "unscored_reason", "decision_basis",
)
FP_FN_CHANGES = {
    ("FP", "TN"): "Fixed FP",
    ("FN", "TP"): "Fixed FN",
    ("TN", "FP"): "New FP",
    ("TP", "FN"): "New FN",
}
COMPARISON_COLUMNS = {
    "change": "Change",
    "case_id": "Case ID",
    "dataset": "Dataset",
    "expected_risk": "Expected",
    "original_probability": "Original P(risk)",
    "post_trained_probability": "Tuned P(risk)",
    "original_predicted_risk": "Original prediction",
    "post_trained_predicted_risk": "Tuned prediction",
    "original_outcome": "Original outcome",
    "post_trained_outcome": "Tuned outcome",
}


def comparison_change(original, tuned):
    if tuned is None:
        return "Original only"
    if original == tuned:
        return "Unchanged"
    if original == "U":
        return "Newly scored"
    if tuned == "U":
        return "Newly unscored"
    return FP_FN_CHANGES[original, tuned]


def pair_evaluation_cases(evidence):
    original_cases = evidence["base"]
    original_ids = {case["case_id"] for case in original_cases}
    if len(original_ids) != len(original_cases):
        raise ValueError("Original evaluation contains duplicate case IDs.")

    tuned_by_id = {}
    if "post-trained" in evidence:
        tuned_cases = evidence["post-trained"]
        tuned_by_id = {case["case_id"]: case for case in tuned_cases}
        if len(tuned_by_id) != len(tuned_cases):
            raise ValueError("Post-trained evaluation contains duplicate case IDs.")
        if tuned_by_id.keys() != original_ids:
            raise ValueError("Original and post-trained evaluations must cover the same case IDs.")

    comparison = []
    for original in original_cases:
        case_id = original["case_id"]
        tuned = tuned_by_id.get(case_id)
        row = {field: original[field] for field in CASE_CONTEXT_FIELDS}
        if tuned is not None and any(tuned[field] != value for field, value in row.items()):
            raise ValueError(f"Case {case_id!r} has different labels, cutoff or source metadata.")
        for prefix, case in (("original", original), ("post_trained", tuned)):
            for field in CASE_PREDICTION_FIELDS:
                row[f"{prefix}_{field}"] = case[field] if case is not None else None
        row["change"] = comparison_change(row["original_outcome"], row["post_trained_outcome"])
        comparison.append(row)
    return comparison


def save_evaluation_case_evidence(split, rows):
    destination = OUTPUT / f"evaluation-{split}-cases.jsonl"
    destination.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=destination.parent) as directory:
        staged = Path(directory) / destination.name
        with staged.open("w", encoding="utf-8") as stream:
            for row in rows:
                stream.write(json.dumps(row) + "\n")
        staged.replace(destination)
    return destination


def show_evaluation_case_changes(change=None, *, limit=10):
    """Show up to limit cases per change. Use limit=None to show every match."""
    changes = tuple(FP_FN_CHANGES.values()) if change is None else (change,)
    table = EVALUATION_CASE_COMPARISON
    groups = [table.loc[table["change"] == name, list(COMPARISON_COLUMNS)] for name in changes]
    print(" · ".join(f"{name} {len(group):,}" for name, group in zip(changes, groups, strict=True)))
    selected = pd.concat(groups if limit is None else [g.head(limit) for g in groups], ignore_index=True)
    if selected.empty:
        return
    print(f"Showing {len(selected):,} of {sum(map(len, groups)):,} matching cases.")
    for field in ("expected_risk", "original_predicted_risk", "post_trained_predicted_risk"):
        selected[field] = selected[field].map({0: "Benign", 1: "Risk"})
    for field in ("original_probability", "post_trained_probability"):
        selected[field] = selected[field].map(str, na_action="ignore")
    selected = selected.astype(object).where(selected.notna(), "—")
    with pd.option_context("display.max_columns", None, "display.max_colwidth", None, "display.max_rows", None):
        display(selected.rename(columns=COMPARISON_COLUMNS))


EVALUATION_CASE_EVIDENCE_PATH = None
EVALUATION_CASE_RECORDS = pair_evaluation_cases(EVALUATION_CASES)
comparison_fields = [
    *CASE_CONTEXT_FIELDS,
    *(f"{model}_{field}" for model in ("original", "post_trained") for field in CASE_PREDICTION_FIELDS),
    "change",
]
EVALUATION_CASE_COMPARISON = pd.DataFrame(EVALUATION_CASE_RECORDS, columns=comparison_fields)
EVALUATION_CASE_EVIDENCE_PATH = save_evaluation_case_evidence(
    EVALUATED["base"]["split"], EVALUATION_CASE_RECORDS
)
print(f"Saved {len(EVALUATION_CASE_COMPARISON):,} case records to {EVALUATION_CASE_EVIDENCE_PATH}.")
if "post-trained" in EVALUATION_CASES:
    show_evaluation_case_changes()
else:
    print("Saved the Original results. Case changes require a post-trained checkpoint.")

### Check risk score separation

Summarizes AUC and score distributions by task and prior dataset exposure. Run after the per-case comparison to see how well the models separate benign and risky samples.

In [ ]:
import pandas as pd
from IPython.display import display


def source_family(source):
    name = source.split("/", 1)[0]
    # Both MPBench cohorts are derived from the same upstream corpus.
    return "mpbench" if name == "mpbench-memory-v1" else name


def roc_auc(probabilities, risk):
    positives = int(risk.sum())
    negatives = len(risk) - positives
    if not positives or not negatives:
        return None
    ranks = probabilities.rank(method="average")
    positive_ranks = ranks[risk].sum()
    return float((positive_ranks - positives * (positives + 1) / 2) / (positives * negatives))


def summarize_score_separation(cases):
    score_columns = ("original_probability", "post_trained_probability")
    paired = cases.dropna(subset=list(score_columns))
    risk = paired["expected_risk"] == 1
    original_auc, tuned_auc = (roc_auc(paired[column], risk) for column in score_columns)

    def pair(values, digits):
        first, second = (f"{value:.{digits}f}" for value in values)
        return first if first == second else f"{first} → {second}"

    def label_stats(mask):
        subset = paired.loc[mask]
        if subset.empty:
            return "—", "—"
        scores = subset.loc[:, list(score_columns)]
        medians = scores.median()
        rates = scores.ge(subset["cutoff"], axis="index").mean() * 100
        return pair(medians, 3), pair(rates, 2)

    risk_p50, rr = label_stats(risk)
    benign_p50, fpr = label_stats(~risk)
    return {
        "Test n": len(cases),
        "Paired n": len(paired),
        "Risk n": int(risk.sum()),
        "Benign n": int((~risk).sum()),
        "Original AUC": original_auc,
        "Tuned AUC": tuned_auc,
        "Δ AUC": tuned_auc - original_auc if tuned_auc is not None and original_auc is not None else None,
        "Risk P50": risk_p50,
        "Benign P50": benign_p50,
        "RR (%)": rr,
        "FPR (%)": fpr,
    }


def score_separation_table(comparison, *, dataset=None):
    sources = {
        split: {source_family(row["source"]) for row in read_split(split)}
        for split in ("train", "validation", "calibration_fit", "calibration_policy")
    }
    development = sources["validation"] | sources["calibration_fit"] | sources["calibration_policy"]

    def source_scope(source):
        name = source_family(source)
        if name in sources["train"]:
            return "Training source"
        if name in development:
            return "Validation / calibration source"
        return "Outside train and dev"

    rows = comparison if dataset is None else comparison.loc[comparison["dataset"] == dataset]
    rows = rows.assign(source_scope=rows["source"].map(source_scope))
    report = []
    for (task, phase), cases in rows.groupby(["task", "phase"], sort=True):
        task_label = f"{TASK_LABELS[task]} ({phase})"
        groups = list(cases.groupby("source_scope", sort=True))
        if len(groups) > 1:
            groups.insert(0, ("All sources", cases))
        for scope, group in groups:
            report.append({
                "Task": task_label,
                "Source": scope,
                **summarize_score_separation(group),
            })
    return pd.DataFrame(report)


def show_score_separation(comparison, *, dataset=None):
    report = score_separation_table(comparison, dataset=dataset).round(3)
    print("Original → Post-trained on cases where both models returned a score.")
    print("AUC compares score ranks. RR and FPR use the recorded cutoff.")
    with pd.option_context("display.max_columns", None):
        display(report.astype(object).where(report.notna(), "—"))
    return report


if EVALUATION_CASE_COMPARISON is None:
    raise RuntimeError("Run evaluation-cases first.")
if EVALUATION_CASE_COMPARISON["post_trained_outcome"].notna().any():
    EVALUATION_SCORE_SEPARATION = show_score_separation(EVALUATION_CASE_COMPARISON)
else:
    EVALUATION_SCORE_SEPARATION = None
    print("Score separation comparison requires both checkpoints.")

### Compare original and tuned models at the FPR budget

Fits a threshold for each model and task × phase on `calibration_policy` using `FPR_BUDGET`. Run with the merged checkpoint to report actual acceptance FPR and RR and flag missing calibration data.

In [ ]:
import csv
import json
import math
from collections import Counter, defaultdict
from datetime import datetime, timezone
from fractions import Fraction
from functools import partial
from itertools import groupby
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
from IPython.display import display

FPR_BUDGET = 0.05


def threshold_at_fpr(paired, model, max_fp):
    def probability(case):
        return case[f"{model}_probability"]

    ordered = sorted(paired, key=probability, reverse=True)
    threshold = math.nextafter(1.0, math.inf)  # Allow every score in [0, 1].
    fp = 0
    for score, tied in groupby(ordered, key=probability):
        labels = Counter(case["expected_risk"] for case in tied)
        fp += labels[0]
        if fp > max_fp:
            break
        if labels[1]:
            # For equal RR, keep the stricter threshold and the lower FPR.
            threshold = score
    return threshold


def compare_fpr_budget(fpr_budget, models, output_root):
    budget = Fraction(str(fpr_budget))
    if not 0 <= budget <= 1:
        raise ValueError("FPR_BUDGET must be a fraction between 0 and 1.")

    reference, candidate = models

    splits = {name: read_split(name) for name in ("calibration_policy", "acceptance")}
    cases = {}
    for split, rows in splits.items():
        records = {
            row["id"]: {
                "split": split,
                "case_id": row["id"],
                "dataset": row["source"].split("/", 1)[0],
                "source": row["source"],
                "task": task_for(row),
                "phase": row["phase"],
                "expected_risk": int(row["label"]),
                "expected_action": row["expected"],
            }
            for row in rows
        }
        if len(records) != len(rows):
            raise ValueError(f"Duplicate case_id in {split}.")
        cases[split] = records

    for name, open_agents in models.items():
        with open_agents() as agents:
            for split, rows in splits.items():
                print(f"{name} · {split} · {len(rows):,} cases", flush=True)
                if not rows:
                    continue
                scored = score_split(agents, rows)
                for row, p, reason in zip(
                    rows, scored["probability"], scored["unscored_reason"], strict=True
                ):
                    case = cases[split][row["id"]]
                    case[f"{name.lower()}_probability"] = p
                    case[f"{name.lower()}_unscored_reason"] = reason

    groups = {split: defaultdict(list) for split in splits}
    for split, records in cases.items():
        for case in records.values():
            case["paired_scored"] = all(
                case[f"{name.lower()}_probability"] is not None for name in models
            )
            groups[split][case["task"], case["phase"]].append(case)

    transitions = {
        ("FP", "TN"): "Fixed FP",
        ("FN", "TP"): "Fixed FN",
        ("TN", "FP"): "New FP",
        ("TP", "FN"): "New FN",
    }
    summary = []
    calibration = groups["calibration_policy"]
    for task, phase in sorted(calibration.keys() | groups["acceptance"].keys()):
        key = task, phase
        cal_cases = calibration.get(key, [])
        paired_cal = [case for case in cal_cases if case["paired_scored"]]
        cal_labels = Counter(case["expected_risk"] for case in paired_cal)
        status = "OK"
        if not cal_cases:
            status = "No cal rows"
        elif not paired_cal:
            status = "No paired cal scores"
        elif not cal_labels[0]:
            status = "No paired cal benign"
        elif not cal_labels[1]:
            status = "No paired cal risk"

        max_fp = int(budget * cal_labels[0])
        thresholds = {
            name: threshold_at_fpr(paired_cal, name.lower(), max_fp) if status == "OK" else None
            for name in models
        }
        for split, split_groups in groups.items():
            group = split_groups.get(key, [])
            paired = [case for case in group if case["paired_scored"]]
            n = Counter(case["expected_risk"] for case in paired)
            stats = {
                "Split": split, "Task": task, "Phase": phase, "Cal status": status,
                "FPR budget (%)": float(100 * budget), "Cal max FP": max_fp if status == "OK" else None,
                "Test n": len(group), "Paired n": len(paired), "Benign n": n[0], "Risk n": n[1],
            }
            for name, threshold in thresholds.items():
                prefix = name.lower()
                for case in group:
                    p = case[f"{prefix}_probability"]
                    predicted = int(p >= threshold) if p is not None and threshold is not None else None
                    case[f"{prefix}_threshold"] = threshold
                    case[f"{prefix}_predicted_risk"] = predicted
                    case[f"{prefix}_outcome"] = (
                        CASE_OUTCOMES[case["expected_risk"], predicted] if predicted is not None else None
                    )

                counts = Counter(case[f"{prefix}_outcome"] for case in paired)
                stats[f"{name} threshold"] = threshold
                stats[f"{name} scored n"] = sum(case[f"{prefix}_probability"] is not None for case in group)
                for outcome in ("TP", "TN", "FP", "FN"):
                    stats[f"{name} {outcome}"] = counts[outcome] if threshold is not None else None
                stats[f"{name} FPR (%)"] = 100 * counts["FP"] / n[0] if threshold is not None and n[0] else None
                stats[f"{name} RR (%)"] = 100 * counts["TP"] / n[1] if threshold is not None and n[1] else None

            reference_rr, candidate_rr = stats[f"{reference} RR (%)"], stats[f"{candidate} RR (%)"]
            stats["Δ RR (pp)"] = (
                candidate_rr - reference_rr
                if reference_rr is not None and candidate_rr is not None else None
            )
            summary.append(stats)

            for case in group:
                case["cal_status"] = status
                case["fpr_budget"] = float(budget)
                outcomes = case[f"{reference.lower()}_outcome"], case[f"{candidate.lower()}_outcome"]
                if status != "OK":
                    case["change"] = "No threshold"
                elif not case["paired_scored"]:
                    case["change"] = "Unscored"
                else:
                    case["change"] = transitions.get(outcomes, "Unchanged")

    if not summary:
        raise ValueError("Both splits are empty.")
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)
    destination = output_root / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    with TemporaryDirectory(dir=output_root) as temporary:
        staged = Path(temporary) / "result"
        staged.mkdir()
        with (staged / "summary.csv").open("w", newline="", encoding="utf-8") as stream:
            writer = csv.DictWriter(stream, fieldnames=list(summary[0]))
            writer.writeheader()
            writer.writerows(summary)
        for split, records in cases.items():
            with (staged / f"{split}-cases.jsonl").open("w", encoding="utf-8") as stream:
                for case in records.values():
                    stream.write(json.dumps(case, ensure_ascii=False, allow_nan=False) + "\n")
        staged.rename(destination)

    return summary, cases, destination


def run_fpr_budget(fpr_budget, tuned_dir):
    tuned_dir = Path(tuned_dir)
    model_export_sizes(tuned_dir)
    original_dir = Path(globals().get("MODEL_DIR") or materialize())
    if original_dir.resolve() == tuned_dir.resolve():
        raise ValueError("Original and tuned checkpoint directories must differ.")
    models = {
        "Original": partial(scoring_agents, original_dir, expected_sha256=BASE["files"]),
        "Tuned": partial(scoring_agents, tuned_dir),
    }
    return compare_fpr_budget(fpr_budget, models, OUTPUT / "fpr-budget")


def show_fpr_summary(frame, model_names, budget):
    threshold_columns = [f"{name} threshold" for name in model_names]
    rate_columns = [f"{name} {rate} (%)" for rate in ("FPR", "RR") for name in model_names]
    columns = [
        "Task", "Phase", "Cal status", "Test n", "Paired n", "Benign n", "Risk n",
        *threshold_columns, *rate_columns, "Δ RR (pp)",
    ]
    for split in ("calibration_policy", "acceptance"):
        print(f"{split} · FPR budget {budget:.2%}")
        table = frame.loc[frame["Split"].eq(split), columns].reset_index(drop=True)
        for column in threshold_columns:
            table[column] = table[column].map(
                lambda t: "—" if pd.isna(t) else ("allow all" if t > 1 else repr(float(t)))
            )
        table = table.round({column: 2 for column in (*rate_columns, "Δ RR (pp)")})
        with pd.option_context("display.max_columns", None):
            display(table.astype(object).where(table.notna(), "—"))


FPR_SUMMARY = FPR_ACCEPTANCE_CASES = FPR_CASES = FPR_OUTPUT_DIR = None
evaluated_tuned_dir = (globals().get("EVALUATED") or {}).get("post-trained", {}).get("modelDir")
FPR_MODEL_DIR = Path(evaluated_tuned_dir) if evaluated_tuned_dir else resolve_tuned_model_dir()
if FPR_MODEL_DIR is not None:
    print(f"FPR-budget evaluation uses {FPR_MODEL_DIR}")
    _fpr_summary, FPR_CASES, FPR_OUTPUT_DIR = run_fpr_budget(FPR_BUDGET, FPR_MODEL_DIR)
    FPR_SUMMARY = pd.DataFrame(_fpr_summary)
    FPR_ACCEPTANCE_CASES = pd.DataFrame(list(FPR_CASES["acceptance"].values()))

    show_fpr_summary(FPR_SUMMARY, ("Original", "Tuned"), FPR_BUDGET)
    print(f"Saved summary and paired case records to {FPR_OUTPUT_DIR}")
else:
    print("FPR-budget evaluation needs a merged checkpoint. Attach a saved Notebook Output containing laya-guardrails/model/.")

### Compare FP16 and INT8 on acceptance

Uses the exported models to compare acceptance FPR and RR at the same calibration budget. Run after INT8 export. The cell saves paired case records and a separate table at the standard 0.5 cutoff.

In [ ]:
def int8_fixed_cutoff_results(cases, acceptance):
    results = {}
    for key, model in (("base", "fp16"), ("post-trained", "int8")):
        scores = {
            field: [cases[row["id"]][f"{model}_{field}"] for row in acceptance]
            for field in ("probability", "unscored_reason")
        }
        evidence = baseline_case_evidence("acceptance", acceptance, scores)
        results[key] = {"by_dataset": baseline_metrics(evidence)}
    return results


INT8_FIXED_CUTOFF_OUTPUT = None
INT8_FPR_SUMMARY = INT8_FPR_CASES = INT8_FPR_OUTPUT_DIR = None
if INT8_MODEL_DIR is not None:
    int8_loaders = {
        "FP16": partial(fp16_cpu_agents, INT8_FP16_DIR),
        "INT8": partial(int8_cpu_agents, INT8_MODEL_DIR),
    }
    int8_summary, INT8_FPR_CASES, INT8_FPR_OUTPUT_DIR = compare_fpr_budget(
        FPR_BUDGET, int8_loaders, INT8_MODEL_DIR.parent / "evaluation",
    )
    INT8_FPR_SUMMARY = pd.DataFrame(int8_summary)
    show_fpr_summary(INT8_FPR_SUMMARY, int8_loaders, FPR_BUDGET)

    fixed_cutoff = int8_fixed_cutoff_results(INT8_FPR_CASES["acceptance"], read_split("acceptance"))
    print(f"FP16 / INT8 · acceptance · risk cutoff {BASELINE_CUTOFF:g}")
    show_evaluation_comparison(fixed_cutoff, labels=("FP16", "INT8"), direction="FP16 → INT8")
    fixed_table = pd.DataFrame(fixed_cutoff["base"]["by_dataset"]).merge(
        pd.DataFrame(fixed_cutoff["post-trained"]["by_dataset"]),
        on=["dataset", "task", "phase"], suffixes=("_fp16", "_int8"), validate="one_to_one",
    )
    fixed_table.insert(3, "cutoff", BASELINE_CUTOFF)
    INT8_FIXED_CUTOFF_OUTPUT = INT8_FPR_OUTPUT_DIR / "fixed-cutoff-acceptance.csv"
    fixed_table.to_csv(INT8_FIXED_CUTOFF_OUTPUT, index=False)
    print(f"Saved FP16 / INT8 summary and paired case records to {INT8_FPR_OUTPUT_DIR}")

### Save the run record

Writes `laya-material.json` with the dataset, checkpoint information, and available evaluation results. Run last and check the file in Kaggle Output.

In [ ]:
training = OUTPUT / "training.json"
material = {
    "datasetId": DATASET["datasetId"],
    "datasetVersion": DATASET["version"],
    "datasetManifestSha256": DATASET["sha256"],
    "datasetRef": DATASET["ref"],
    "splits": MANIFEST["splits"],
    "baseCheckpoint": BASE,
    "recipe": TRAINING,
    "runtimeBudgets": RUNTIME,
    "training": json.loads(training.read_text()) if training.is_file() else None,
    "originalBaseline": globals().get("ORIGINAL_BASELINE"),
    "originalBaselineCaseEvidence": (
        str(BASELINE_EVIDENCE_PATH) if "BASELINE_EVIDENCE_PATH" in globals() else None
    ),
    "huggingFaceUpload": globals().get("HF_UPLOAD_RECEIPT"),
    "int8Model": str(INT8_MODEL_DIR / "model.onnx") if globals().get("INT8_MODEL_DIR") else None,
    "int8FprEvidence": str(INT8_FPR_OUTPUT_DIR) if globals().get("INT8_FPR_OUTPUT_DIR") else None,
    "int8FixedCutoffAcceptance": (
        str(INT8_FIXED_CUTOFF_OUTPUT) if globals().get("INT8_FIXED_CUTOFF_OUTPUT") else None
    ),
    "evaluation": globals().get("EVALUATED"),
    "evaluationCaseEvidence": (
        str(EVALUATION_CASE_EVIDENCE_PATH) if globals().get("EVALUATION_CASE_EVIDENCE_PATH") else None
    ),
}
destination = WORK / "laya-material.json"
destination.write_text(json.dumps(material, indent=2, sort_keys=True))
for split, entry in material["splits"].items():
    print(f"{split:<18} {entry['count']:>7,} rows  sha256 {entry['sha256'][:12]}")
print("training  :", "recorded" if material["training"] else "not run")
print("evaluation:", "recorded" if material["evaluation"] else "not run")
print(f"material record: {destination}")